In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

metadata_path = "/content/data/dataverse_files/HAM10000_metadata"
df = pd.read_csv(metadata_path)

label_map = {
    "nv": 0,
    "mel": 1,
    "bcc": 2,
    "akiec": 3,
    "bkl": 4,
    "df": 5,
    "vasc": 6
}

df["label"] = df["dx"].map(label_map)

print(df["dx"].value_counts())

In [ ]:
print("Unique lesions:", df["lesion_id"].nunique())
print("Total images :", len(df))

In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T


class HAM10000FullDataset(Dataset):
    def __init__(self, image_dir, mask_dir, df, image_size=224):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        # index by image_id for fast lookup
        self.df = df.set_index("image_id")

        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
        ])

        self.mask_transform = T.Compose([
            T.Resize((image_size, image_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
        ])

        self.samples = []

        for img_path in self.image_dir.glob("*.jpg"):
            image_id = img_path.stem

            # mask
            mask_path = self.mask_dir / f"{image_id}_segmentation.png"

            # label
            if image_id in self.df.index and mask_path.exists():
                label = self.df.loc[image_id]["label"]
                self.samples.append((img_path, mask_path, label, image_id))

        print(f"Total valid samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label, image_id = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = self.image_transform(image)
        mask = self.mask_transform(mask)

        mask = (mask > 0.5).float()

        return {
            "image": image,        # [3, H, W]
            "mask": mask,          # [1, H, W]
            "label": torch.tensor(label),
            "image_id": image_id
        }

In [ ]:
image_dir = "/content/data/dataverse_files/HAM10000_images_combined_600x450"
mask_dir  = "/content/data/dataverse_files/HAM10000_segmentations_lesion_tschandl"

dataset = HAM10000FullDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    df=df,
    image_size=224
)

print(len(dataset))

In [ ]:
!pip install scikit-image scipy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
from scipy.ndimage import binary_erosion
from skimage.measure import find_contours

# ---------------------------------------------------
# class names
# ---------------------------------------------------
label_names = {
    0: "nv",
    1: "mel",
    2: "bcc",
    3: "akiec",
    4: "bkl",
    5: "df",
    6: "vasc",
}

pretty_names = {
    "nv": "Melanocytic Nevi",
    "mel": "Melanoma",
    "bcc": "Basal Cell Carcinoma",
    "akiec": "Actinic Keratoses / IEC",
    "bkl": "Benign Keratosis",
    "df": "Dermatofibroma",
    "vasc": "Vascular Lesion",
}

# ---------------------------------------------------
# get one sample per class
# ---------------------------------------------------
def get_one_sample_per_class(dataset):
    chosen = {}
    for i in range(len(dataset)):
        sample = dataset[i]
        label = int(sample["label"])
        if label not in chosen:
            chosen[label] = sample
        if len(chosen) == len(label_names):
            break
    return chosen

# ---------------------------------------------------
# create nicer overlay
# ---------------------------------------------------
def make_publication_overlay(image_np, mask_np, alpha_fill=0.26):
    """
    image_np: H x W x 3 in [0,1]
    mask_np : H x W binary
    """
    image_np = np.clip(image_np, 0, 1)
    mask_np = (mask_np > 0).astype(np.uint8)

    overlay = image_np.copy()

    # warm amber fill
    lesion_color = np.array([1.00, 0.62, 0.18])
    mask_bool = mask_np.astype(bool)
    overlay[mask_bool] = (
        (1 - alpha_fill) * overlay[mask_bool] + alpha_fill * lesion_color
    )

    return np.clip(overlay, 0, 1)

# ---------------------------------------------------
# contour plotting
# ---------------------------------------------------
def draw_smooth_contours(ax, mask_np, color="#5EF2FF", linewidth=2.2):
    mask_np = (mask_np > 0).astype(np.uint8)
    contours = find_contours(mask_np, level=0.5)

    for contour in contours:
        ax.plot(
            contour[:, 1],
            contour[:, 0],
            color=color,
            linewidth=linewidth,
            solid_capstyle="round",
        )

# ---------------------------------------------------
# pretty plotting
# ---------------------------------------------------
def plot_publication_style_samples(dataset, save_path=None):
    samples = get_one_sample_per_class(dataset)
    present_labels = sorted(samples.keys())
    n = len(present_labels)

    fig, axes = plt.subplots(n, 3, figsize=(15, 4.2 * n), facecolor="#0E1117")
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    fig.patch.set_facecolor("#0E1117")

    col_titles = ["Dermoscopic Image", "Ground Truth Mask", "Segmentation Overlay"]
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(
            t,
            fontsize=15,
            color="white",
            weight="bold",
            pad=14
        )

    for row, label in enumerate(present_labels):
        sample = samples[label]

        image = sample["image"].permute(1, 2, 0).cpu().numpy()
        mask = sample["mask"].squeeze(0).cpu().numpy()
        mask_bin = (mask > 0.5).astype(np.uint8)

        overlay = make_publication_overlay(image, mask_bin, alpha_fill=0.28)

        code = label_names[label]
        class_name = pretty_names[code]

        # -------- original image --------
        ax = axes[row, 0]
        ax.imshow(image)
        ax.set_facecolor("#0E1117")
        ax.axis("off")

        # -------- mask --------
        ax = axes[row, 1]
        ax.imshow(mask_bin, cmap="gray")
        ax.set_facecolor("#0E1117")
        ax.axis("off")

        # -------- overlay --------
        ax = axes[row, 2]
        ax.imshow(overlay)
        draw_smooth_contours(ax, mask_bin, color="#67E8F9", linewidth=2.4)
        ax.set_facecolor("#0E1117")
        ax.axis("off")

        # row label at left side
        axes[row, 0].text(
            -0.08, 0.5,
            class_name,
            transform=axes[row, 0].transAxes,
            fontsize=13,
            color="white",
            weight="bold",
            va="center",
            ha="right",
            rotation=90
        )

        # subtle panel borders
        for c in range(3):
            axes[row, c].add_patch(
                patches.Rectangle(
                    (0, 0),
                    1,
                    1,
                    transform=axes[row, c].transAxes,
                    fill=False,
                    linewidth=1.2,
                    edgecolor=(1, 1, 1, 0.14),
                    clip_on=False
                )
            )

    fig.suptitle(
        "HAM10000 Segmentation Diagnostic ",
        fontsize=20,
        color="white",
        weight="bold",
        y=0.995
    )


    plt.tight_layout(rect=[0.04, 0.02, 1, 0.975])

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())

    plt.show()

# ---------------------------------------------------
# run
# ---------------------------------------------------
plot_publication_style_samples(dataset, save_path="ham10000_classwise_overlay.png")

In [ ]:
import numpy as np
import pandas as pd

def split_by_lesion(df, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    np.random.seed(seed)

    # unique lesions
    lesion_ids = df["lesion_id"].unique()
    np.random.shuffle(lesion_ids)

    n = len(lesion_ids)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train_ids = lesion_ids[:n_train]
    val_ids   = lesion_ids[n_train:n_train + n_val]
    test_ids  = lesion_ids[n_train + n_val:]

    # create splits
    df_train = df[df["lesion_id"].isin(train_ids)].copy()
    df_val   = df[df["lesion_id"].isin(val_ids)].copy()
    df_test  = df[df["lesion_id"].isin(test_ids)].copy()

    return df_train, df_val, df_test

In [ ]:
df_train, df_val, df_test = split_by_lesion(df)

print("Train:", len(df_train))
print("Val  :", len(df_val))
print("Test :", len(df_test))

In [ ]:
train_lesions = set(df_train["lesion_id"])
val_lesions   = set(df_val["lesion_id"])
test_lesions  = set(df_test["lesion_id"])

print("Train ∩ Val:", len(train_lesions & val_lesions))
print("Train ∩ Test:", len(train_lesions & test_lesions))
print("Val ∩ Test:", len(val_lesions & test_lesions))

In [ ]:
def print_distribution(df, name):
    print(f"\n{name} distribution:")
    print(df["dx"].value_counts(normalize=True))

print_distribution(df_train, "Train")
print_distribution(df_val, "Val")
print_distribution(df_test, "Test")

In [ ]:
image_dir = "/content/data/dataverse_files/HAM10000_images_combined_600x450"
mask_dir  = "/content/data/dataverse_files/HAM10000_segmentations_lesion_tschandl"

train_dataset = HAM10000FullDataset(image_dir, mask_dir, df_train)
val_dataset   = HAM10000FullDataset(image_dir, mask_dir, df_val)
test_dataset  = HAM10000FullDataset(image_dir, mask_dir, df_test)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
batch = next(iter(train_loader))

print("Image:", batch["image"].shape)
print("Mask :", batch["mask"].shape)
print("Label:", batch["label"].shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=1,
    ignore_mismatched_sizes=True
).to(device)

print("Model loaded")

In [ ]:
from monai.losses import DiceLoss

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss(sigmoid=True)

def compute_segformer_loss(logits, masks):
    """
    logits: [B, 1, h, w]
    masks : [B, 1, H, W]
    """
    logits = F.interpolate(
        logits,
        size=masks.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    loss = bce_loss(logits, masks) + dice_loss(logits, masks)
    return loss, logits

def dice_score_from_logits(logits, masks, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    preds = preds.view(preds.size(0), -1)
    masks = masks.view(masks.size(0), -1)

    intersection = (preds * masks).sum(dim=1)
    union = preds.sum(dim=1) + masks.sum(dim=1)

    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean().item()

def iou_score_from_logits(logits, masks, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    preds = preds.view(preds.size(0), -1)
    masks = masks.view(masks.size(0), -1)

    intersection = (preds * masks).sum(dim=1)
    union = preds.sum(dim=1) + masks.sum(dim=1) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

In [ ]:
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    n = 0

    pbar = tqdm(loader, desc="Train", leave=False)
    for batch in pbar:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=images)
        logits = outputs.logits   # [B, 1, h, w]

        loss, logits = compute_segformer_loss(logits, masks)
        loss.backward()
        optimizer.step()

        dice = dice_score_from_logits(logits.detach(), masks)
        iou = iou_score_from_logits(logits.detach(), masks)

        total_loss += loss.item()
        total_dice += dice
        total_iou += iou
        n += 1

        pbar.set_postfix({
            "loss": f"{total_loss/n:.4f}",
            "dice": f"{total_dice/n:.4f}",
            "iou": f"{total_iou/n:.4f}",
        })

    return total_loss / n, total_dice / n, total_iou / n


@torch.no_grad()
def validate_one_epoch(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    n = 0

    pbar = tqdm(loader, desc="Val", leave=False)
    for batch in pbar:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        outputs = model(pixel_values=images)
        logits = outputs.logits

        loss, logits = compute_segformer_loss(logits, masks)

        dice = dice_score_from_logits(logits, masks)
        iou = iou_score_from_logits(logits, masks)

        total_loss += loss.item()
        total_dice += dice
        total_iou += iou
        n += 1

        pbar.set_postfix({
            "val_loss": f"{total_loss/n:.4f}",
            "val_dice": f"{total_dice/n:.4f}",
            "val_iou": f"{total_iou/n:.4f}",
        })

    return total_loss / n, total_dice / n, total_iou / n

In [ ]:
NUM_EPOCHS = 20
best_val_dice = -1
save_path = "best_segformer_ham10000.pth"

history = {
    "train_loss": [],
    "train_dice": [],
    "train_iou": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": [],
}

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")

    train_loss, train_dice, train_iou = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_dice, val_iou = validate_one_epoch(model, val_loader, device)

    history["train_loss"].append(train_loss)
    history["train_dice"].append(train_dice)
    history["train_iou"].append(train_iou)

    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    print(f"Train | Loss: {train_loss:.4f}  Dice: {train_dice:.4f}  IoU: {train_iou:.4f}")
    print(f"Val   | Loss: {val_loss:.4f}  Dice: {val_dice:.4f}  IoU: {val_iou:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model to {save_path}")

print("Best val dice:", best_val_dice)

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device))

test_loss, test_dice, test_iou = validate_one_epoch(model, test_loader, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Dice: {test_dice:.4f}")
print(f"Test IoU : {test_iou:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history["train_dice"], label="Train Dice")
plt.plot(history["val_dice"], label="Val Dice")
plt.title("Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history["train_iou"], label="Train IoU")
plt.plot(history["val_iou"], label="Val IoU")
plt.title("IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()
plt.show()

In [ ]:
@torch.no_grad()
def show_segformer_predictions(model, loader, device, num_samples=4):
    model.eval()

    batch = next(iter(loader))
    images = batch["image"].to(device)
    masks = batch["mask"].to(device)

    outputs = model(pixel_values=images)
    logits = outputs.logits
    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

    preds = (torch.sigmoid(logits) > 0.5).float()

    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()

    num_samples = min(num_samples, images.size(0))

    plt.figure(figsize=(12, 4 * num_samples))
    for i in range(num_samples):
        img = images[i].permute(1, 2, 0).numpy()
        gt = masks[i].squeeze(0).numpy()
        pr = preds[i].squeeze(0).numpy()

        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(gt, cmap="gray")
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(img)
        plt.imshow(pr, cmap="jet", alpha=0.35)
        plt.title("Prediction Overlay")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_segformer_predictions(model, test_loader, device, num_samples=4)

In [ ]:
# =========================================================
# TEST SET ANALYSIS FOR SEGMENTATION
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from IPython.display import display


# ---------------------------------------------------------
# 1. METRIC FUNCTIONS
# ---------------------------------------------------------
def compute_binary_metrics_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    """
    logits : [B, 1, H, W]
    targets: [B, 1, H, W]
    returns dict of batch metrics and per-sample metrics
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds_f = preds.view(preds.size(0), -1)
    targs_f = targets.view(targets.size(0), -1)

    tp = (preds_f * targs_f).sum(dim=1)
    fp = (preds_f * (1 - targs_f)).sum(dim=1)
    fn = ((1 - preds_f) * targs_f).sum(dim=1)
    tn = ((1 - preds_f) * (1 - targs_f)).sum(dim=1)

    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)

    metrics = {
        "dice": dice.cpu().numpy(),
        "iou": iou.cpu().numpy(),
        "precision": precision.cpu().numpy(),
        "recall": recall.cpu().numpy(),
        "accuracy": accuracy.cpu().numpy(),
    }
    return metrics, preds, probs


# ---------------------------------------------------------
# 2. EVALUATE TEST SET
# ---------------------------------------------------------
@torch.no_grad()
def evaluate_segmentation_testset(model, loader, device, threshold=0.5):
    model.eval()

    rows = []
    saved_samples = []

    for batch_idx, batch in enumerate(loader):
        images = batch["image"].to(device)          # [B, 3, H, W]
        masks = batch["mask"].to(device)            # [B, 1, H, W]
        labels = batch["label"].cpu().numpy()

        image_ids = batch.get("image_id", [f"sample_{batch_idx}_{i}" for i in range(images.size(0))])

        outputs = model(pixel_values=images)
        logits = outputs.logits

        # upsample logits to GT size
        logits = F.interpolate(
            logits,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        metrics, preds, probs = compute_binary_metrics_from_logits(logits, masks, threshold=threshold)

        preds_cpu = preds.cpu()
        probs_cpu = probs.cpu()
        images_cpu = images.cpu()
        masks_cpu = masks.cpu()

        for i in range(images.size(0)):
            gt_mask = masks_cpu[i, 0].numpy()
            pred_mask = preds_cpu[i, 0].numpy()
            prob_map = probs_cpu[i, 0].numpy()
            image_np = images_cpu[i].permute(1, 2, 0).numpy()

            gt_area = float(gt_mask.sum())
            pred_area = float(pred_mask.sum())

            row = {
                "image_id": image_ids[i],
                "class_label": int(labels[i]),
                "dice": float(metrics["dice"][i]),
                "iou": float(metrics["iou"][i]),
                "precision": float(metrics["precision"][i]),
                "recall": float(metrics["recall"][i]),
                "accuracy": float(metrics["accuracy"][i]),
                "gt_area_pixels": gt_area,
                "pred_area_pixels": pred_area,
                "abs_area_error": abs(pred_area - gt_area),
            }
            rows.append(row)

            saved_samples.append({
                "image_id": image_ids[i],
                "class_label": int(labels[i]),
                "image": image_np,
                "gt_mask": gt_mask,
                "pred_mask": pred_mask,
                "prob_map": prob_map,
                "dice": float(metrics["dice"][i]),
                "iou": float(metrics["iou"][i]),
                "precision": float(metrics["precision"][i]),
                "recall": float(metrics["recall"][i]),
                "accuracy": float(metrics["accuracy"][i]),
            })

    results_df = pd.DataFrame(rows)

    summary_df = pd.DataFrame([{
        "Dice_mean": results_df["dice"].mean(),
        "Dice_std": results_df["dice"].std(),
        "IoU_mean": results_df["iou"].mean(),
        "IoU_std": results_df["iou"].std(),
        "Precision_mean": results_df["precision"].mean(),
        "Recall_mean": results_df["recall"].mean(),
        "Accuracy_mean": results_df["accuracy"].mean(),
        "Mean_abs_area_error": results_df["abs_area_error"].mean(),
        "Num_samples": len(results_df),
    }])

    return results_df, summary_df, saved_samples


# ---------------------------------------------------------
# 3. RUN TEST EVALUATION
# ---------------------------------------------------------
results_df, summary_df, saved_samples = evaluate_segmentation_testset(
    model=model,
    loader=test_loader,
    device=device,
    threshold=0.5
)

print("===== SUMMARY TABLE =====")
display(summary_df)

print("\n===== FIRST 10 SAMPLE-WISE RESULTS =====")
display(results_df.head(10))

# optional save
results_df.to_csv("segmentation_test_results.csv", index=False)
summary_df.to_csv("segmentation_test_summary.csv", index=False)

print("Saved:")
print(" - segmentation_test_results.csv")
print(" - segmentation_test_summary.csv")


# ---------------------------------------------------------
# 4. CLASS-WISE ANALYSIS TABLE
# ---------------------------------------------------------
inv_label_map = {
    0: "nv",
    1: "mel",
    2: "bcc",
    3: "akiec",
    4: "bkl",
    5: "df",
    6: "vasc",
}

results_df["class_name"] = results_df["class_label"].map(inv_label_map)

classwise_df = (
    results_df
    .groupby("class_name")[["dice", "iou", "precision", "recall", "accuracy", "abs_area_error"]]
    .agg(["mean", "std", "count"])
)

print("\n===== CLASS-WISE PERFORMANCE =====")
display(classwise_df)


# ---------------------------------------------------------
# 5. NICE OVERLAY HELPERS
# ---------------------------------------------------------
def draw_mask_contour(ax, mask, color="cyan", linewidth=2.0):
    import matplotlib.pyplot as plt
    from skimage.measure import find_contours

    contours = find_contours(mask.astype(np.uint8), 0.5)
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color=color, linewidth=linewidth)


def make_overlay(image, mask, color=(1.0, 0.55, 0.0), alpha=0.30):
    """
    image: HxWx3
    mask : HxW binary
    """
    overlay = image.copy()
    mask_bool = mask.astype(bool)
    color_arr = np.array(color).reshape(1, 1, 3)
    overlay[mask_bool] = (1 - alpha) * overlay[mask_bool] + alpha * color_arr.reshape(3,)
    return np.clip(overlay, 0, 1)


# ---------------------------------------------------------
# 6. VISUALIZE BEST / MEDIAN / WORST CASES
# ---------------------------------------------------------
def plot_selected_cases(samples, mode="best", n=4):
    if mode == "best":
        chosen = sorted(samples, key=lambda x: x["dice"], reverse=True)[:n]
        title = f"Best {n} Test Predictions"
    elif mode == "worst":
        chosen = sorted(samples, key=lambda x: x["dice"])[:n]
        title = f"Worst {n} Test Predictions"
    elif mode == "median":
        sorted_samples = sorted(samples, key=lambda x: x["dice"])
        mid = len(sorted_samples) // 2
        half = n // 2
        chosen = sorted_samples[max(0, mid - half): min(len(sorted_samples), mid + half)]
        title = f"Median-Quality Test Predictions"
    else:
        raise ValueError("mode must be one of: best, worst, median")

    rows = len(chosen)
    fig, axes = plt.subplots(rows, 4, figsize=(16, 4 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, axis=0)

    fig.suptitle(title, fontsize=18, weight="bold")

    for r, sample in enumerate(chosen):
        image = sample["image"]
        gt = sample["gt_mask"]
        pred = sample["pred_mask"]
        prob = sample["prob_map"]

        gt_overlay = make_overlay(image, gt, color=(0.0, 1.0, 0.9), alpha=0.28)
        pred_overlay = make_overlay(image, pred, color=(1.0, 0.6, 0.0), alpha=0.28)

        # original
        axes[r, 0].imshow(image)
        axes[r, 0].set_title(f"Image\n{sample['image_id']}")
        axes[r, 0].axis("off")

        # GT overlay
        axes[r, 1].imshow(gt_overlay)
        draw_mask_contour(axes[r, 1], gt, color="#54F7E3", linewidth=2.2)
        axes[r, 1].set_title("Ground Truth Overlay")
        axes[r, 1].axis("off")

        # prediction overlay
        axes[r, 2].imshow(pred_overlay)
        draw_mask_contour(axes[r, 2], pred, color="#FFD166", linewidth=2.2)
        axes[r, 2].set_title(
            f"Prediction Overlay\nDice={sample['dice']:.3f}, IoU={sample['iou']:.3f}"
        )
        axes[r, 2].axis("off")

        # probability map
        im = axes[r, 3].imshow(prob, cmap="magma")
        axes[r, 3].set_title("Probability Map")
        axes[r, 3].axis("off")
        plt.colorbar(im, ax=axes[r, 3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


plot_selected_cases(saved_samples, mode="best", n=4)
plot_selected_cases(saved_samples, mode="median", n=4)
plot_selected_cases(saved_samples, mode="worst", n=4)


# ---------------------------------------------------------
# 7. HISTOGRAM OF TEST DICE
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.hist(results_df["dice"], bins=20, edgecolor="black")
plt.title("Distribution of Dice Scores on the Test Set", fontsize=14, weight="bold")
plt.xlabel("Dice Score")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 8. SCATTER: LESION SIZE VS PERFORMANCE
# ---------------------------------------------------------
plt.figure(figsize=(7, 5))
plt.scatter(results_df["gt_area_pixels"], results_df["dice"], alpha=0.65)
plt.title("Ground-Truth Lesion Size vs Dice Score", fontsize=14, weight="bold")
plt.xlabel("Ground-Truth Mask Area (pixels)")
plt.ylabel("Dice Score")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 9. TOP FAILURES TABLE
# ---------------------------------------------------------
worst_cases_df = results_df.sort_values("dice", ascending=True).head(10)
best_cases_df = results_df.sort_values("dice", ascending=False).head(10)

print("\n===== WORST 10 TEST CASES =====")
display(worst_cases_df)

print("\n===== BEST 10 TEST CASES =====")
display(best_cases_df)


# ---------------------------------------------------------
# 10. OPTIONAL: CLASS-WISE BOXPLOT
# ---------------------------------------------------------
plt.figure(figsize=(10, 5))
results_df.boxplot(column="dice", by="class_name", grid=False, rot=30)
plt.title("Dice Score by Diagnostic Class", fontsize=14, weight="bold")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Dice Score")
plt.tight_layout()
plt.show()

In [ ]:
def make_clean_summary_table(summary_df):
    df = summary_df.copy()

    df["Dice"] = df["Dice_mean"].round(3).astype(str) + " ± " + df["Dice_std"].round(3).astype(str)
    df["IoU"]  = df["IoU_mean"].round(3).astype(str) + " ± " + df["IoU_std"].round(3).astype(str)

    df_clean = df[[
        "Dice", "IoU", "Precision_mean", "Recall_mean", "Accuracy_mean", "Mean_abs_area_error"
    ]].rename(columns={
        "Precision_mean": "Precision",
        "Recall_mean": "Recall",
        "Accuracy_mean": "Accuracy",
        "Mean_abs_area_error": "Area Error"
    })

    print("\n===== CLEAN SUMMARY TABLE =====\n")
    display(df_clean)

make_clean_summary_table(summary_df)

In [ ]:
plt.figure(figsize=(8,5))

values = results_df["dice"].values
values_sorted = np.sort(values)

plt.plot(values_sorted, linewidth=2.5)
plt.fill_between(range(len(values_sorted)), values_sorted, alpha=0.2)

plt.title("Sorted Dice Scores (Model Stability)", fontsize=14, weight="bold")
plt.xlabel("Samples (sorted)")
plt.ylabel("Dice Score")

plt.axhline(y=values.mean(), linestyle="--", label="Mean Dice")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from skimage.measure import find_contours
import pandas as pd

# =========================================================
# 1. HELPER FUNCTIONS
# =========================================================

def draw_contour(ax, mask, color="cyan", linewidth=2.0):
    contours = find_contours(mask.astype(np.uint8), 0.5)
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color=color, linewidth=linewidth)


def make_soft_overlay(image, mask, color=(1.0, 0.6, 0.0), alpha=0.28):
    overlay = image.copy()
    mask_bool = mask.astype(bool)
    color_arr = np.array(color)
    overlay[mask_bool] = (1 - alpha) * overlay[mask_bool] + alpha * color_arr
    return np.clip(overlay, 0, 1)


def make_error_map(gt, pred):
    """
    Returns:
    0 = background correct
    1 = true positive
    2 = false positive
    3 = false negative
    """
    gt = gt.astype(bool)
    pred = pred.astype(bool)

    err = np.zeros_like(gt, dtype=np.uint8)
    err[(gt == 1) & (pred == 1)] = 1   # TP
    err[(gt == 0) & (pred == 1)] = 2   # FP
    err[(gt == 1) & (pred == 0)] = 3   # FN
    return err


# =========================================================
# 2. BEST / MEDIAN / WORST VISUAL PANELS
# =========================================================

def plot_case_panel(samples, mode="best", n=4):
    if mode == "best":
        chosen = sorted(samples, key=lambda x: x["dice"], reverse=True)[:n]
        title = f"Best {n} Segmentation Cases"
    elif mode == "worst":
        chosen = sorted(samples, key=lambda x: x["dice"])[:n]
        title = f"Worst {n} Segmentation Cases"
    elif mode == "median":
        sorted_samples = sorted(samples, key=lambda x: x["dice"])
        mid = len(sorted_samples) // 2
        start = max(0, mid - n // 2)
        chosen = sorted_samples[start:start+n]
        title = f"Median-Quality Segmentation Cases"
    else:
        raise ValueError("mode must be best, median, or worst")

    fig, axes = plt.subplots(len(chosen), 5, figsize=(20, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, 0)

    fig.patch.set_facecolor("#0f1117")
    fig.suptitle(title, fontsize=20, color="white", weight="bold", y=0.995)

    for r, s in enumerate(chosen):
        img = s["image"]
        gt = s["gt_mask"].astype(np.uint8)
        pred = s["pred_mask"].astype(np.uint8)
        prob = s["prob_map"]

        gt_overlay = make_soft_overlay(img, gt, color=(0.0, 0.95, 0.85), alpha=0.25)
        pred_overlay = make_soft_overlay(img, pred, color=(1.0, 0.65, 0.10), alpha=0.25)

        err = make_error_map(gt, pred)

        # 1 image
        ax = axes[r, 0]
        ax.imshow(img)
        ax.set_title(f"Image\n{s['image_id']}", color="white", fontsize=12, weight="bold")
        ax.axis("off")
        ax.set_facecolor("#0f1117")

        # 2 gt
        ax = axes[r, 1]
        ax.imshow(gt_overlay)
        draw_contour(ax, gt, color="#59F0E6", linewidth=2.2)
        ax.set_title("Ground Truth", color="white", fontsize=12, weight="bold")
        ax.axis("off")
        ax.set_facecolor("#0f1117")

        # 3 pred
        ax = axes[r, 2]
        ax.imshow(pred_overlay)
        draw_contour(ax, pred, color="#FFD166", linewidth=2.2)
        ax.set_title(
            f"Prediction\nDice={s['dice']:.3f} | IoU={s['iou']:.3f}",
            color="white", fontsize=12, weight="bold"
        )
        ax.axis("off")
        ax.set_facecolor("#0f1117")

        # 4 error map
        ax = axes[r, 3]
        ax.imshow(img)
        cmap = ListedColormap([
            (0, 0, 0, 0.0),      # correct background invisible
            (0.1, 1.0, 0.6, 0.65),  # TP greenish
            (1.0, 0.4, 0.1, 0.75),  # FP orange
            (1.0, 0.1, 0.25, 0.75), # FN red
        ])
        ax.imshow(err, cmap=cmap, interpolation="nearest")
        ax.set_title("Error Map\nTP / FP / FN", color="white", fontsize=12, weight="bold")
        ax.axis("off")
        ax.set_facecolor("#0f1117")

        # 5 confidence
        ax = axes[r, 4]
        im = ax.imshow(prob, cmap="magma")
        ax.set_title(
            f"Confidence Map\nP={s['precision']:.3f}, R={s['recall']:.3f}",
            color="white", fontsize=12, weight="bold"
        )
        ax.axis("off")
        ax.set_facecolor("#0f1117")
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()


# =========================================================
# 3. DICE-RANKED PERFORMANCE CURVE
# =========================================================

def plot_ranked_dice_curve(results_df):
    df = results_df.sort_values("dice").reset_index(drop=True)

    plt.figure(figsize=(10, 5))
    plt.plot(df.index, df["dice"], linewidth=2.5)
    plt.fill_between(df.index, df["dice"], alpha=0.20)

    plt.axhline(df["dice"].mean(), linestyle="--", linewidth=2, label=f"Mean Dice = {df['dice'].mean():.3f}")
    plt.axhline(df["dice"].median(), linestyle=":", linewidth=2, label=f"Median Dice = {df['dice'].median():.3f}")

    plt.title("Ranked Dice Scores Across the Test Set", fontsize=15, weight="bold")
    plt.xlabel("Test Samples (sorted from worst to best)")
    plt.ylabel("Dice Score")
    plt.legend()
    plt.tight_layout()
    plt.show()


# =========================================================
# 4. CLASS-WISE DISTRIBUTION
# =========================================================

def plot_classwise_distribution(results_df):
    class_order = sorted(results_df["class_name"].dropna().unique())

    data = [results_df.loc[results_df["class_name"] == c, "dice"].values for c in class_order]

    plt.figure(figsize=(10, 5))
    parts = plt.violinplot(data, showmeans=True, showmedians=True, widths=0.9)

    for pc in parts['bodies']:
        pc.set_alpha(0.35)

    plt.xticks(np.arange(1, len(class_order) + 1), class_order, rotation=25)
    plt.ylabel("Dice Score")
    plt.title("Class-wise Distribution of Dice Scores", fontsize=15, weight="bold")
    plt.tight_layout()
    plt.show()


# =========================================================
# 5. LESION SIZE VS PERFORMANCE WITH FAILURE HIGHLIGHTING
# =========================================================

def plot_size_vs_performance(results_df):
    df = results_df.copy()
    low_thresh = df["dice"].quantile(0.1)

    plt.figure(figsize=(8, 5))

    # normal cases
    normal = df[df["dice"] > low_thresh]
    failures = df[df["dice"] <= low_thresh]

    plt.scatter(
        normal["gt_area_pixels"],
        normal["dice"],
        alpha=0.45,
        label="Normal cases"
    )

    plt.scatter(
        failures["gt_area_pixels"],
        failures["dice"],
        alpha=0.9,
        marker="x",
        s=70,
        label="Low-Dice failures"
    )

    plt.title("Lesion Size vs Dice Score", fontsize=15, weight="bold")
    plt.xlabel("Ground-truth lesion area (pixels)")
    plt.ylabel("Dice score")
    plt.legend()
    plt.tight_layout()
    plt.show()


# =========================================================
# 6. CONFUSION-STYLE ERROR BAR
# =========================================================

def plot_fp_fn_area_analysis(results_df):
    plt.figure(figsize=(8, 5))
    plt.scatter(
        results_df["abs_area_error"],
        results_df["dice"],
        alpha=0.55
    )
    plt.title("Area Error vs Dice Score", fontsize=15, weight="bold")
    plt.xlabel("Absolute lesion area error (pixels)")
    plt.ylabel("Dice score")
    plt.tight_layout()
    plt.show()


# =========================================================
# 7. RUN ALL
# =========================================================

plot_case_panel(saved_samples, mode="best", n=4)
plot_case_panel(saved_samples, mode="median", n=4)
plot_case_panel(saved_samples, mode="worst", n=4)

#lot_ranked_dice_curve(results_df)
#plot_classwise_distribution(results_df)
#plot_size_vs_performance(results_df)
#plot_fp_fn_area_analysis(results_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from skimage.measure import find_contours

def draw_contours(ax, mask, color="lime", linewidth=2):
    contours = find_contours(mask.astype(np.uint8), 0.5)
    for c in contours:
        ax.plot(c[:, 1], c[:, 0], color=color, linewidth=linewidth)

def boundary_error_map(gt_mask, pred_mask):
    """
    Positive values: outside GT
    Negative values: inside GT / missed regions
    """
    gt = gt_mask.astype(bool)
    pred = pred_mask.astype(bool)

    gt_dist_out = distance_transform_edt(~gt)
    gt_dist_in = distance_transform_edt(gt)

    signed_gt_dist = gt_dist_out.astype(np.float32)
    signed_gt_dist[gt] = -gt_dist_in[gt]

    # evaluate signed GT distance on predicted mask region and neighborhood
    err = np.zeros_like(signed_gt_dist, dtype=np.float32)

    # false positives -> positive error
    fp = pred & (~gt)
    err[fp] = signed_gt_dist[fp]

    # false negatives -> negative error
    fn = gt & (~pred)
    err[fn] = signed_gt_dist[fn]

    return err

def uncertainty_map(prob_map):
    """
    highest uncertainty around 0.5
    """
    return 1.0 - np.abs(prob_map - 0.5) * 2.0

def plot_boundary_uncertainty_cases(saved_samples, indices):
    n = len(indices)
    fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)

    for r, idx in enumerate(indices):
        s = saved_samples[idx]
        img = s["image"]
        gt = s["gt_mask"].astype(np.uint8)
        pred = s["pred_mask"].astype(np.uint8)
        prob = s["prob_map"]

        berr = boundary_error_map(gt, pred)
        unc = uncertainty_map(prob)

        # image
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"Image\n{s['image_id']}", weight="bold")
        axes[r, 0].axis("off")

        # gt contour
        axes[r, 1].imshow(img)
        draw_contours(axes[r, 1], gt, color="cyan", linewidth=2.2)
        axes[r, 1].set_title("GT Contour", weight="bold")
        axes[r, 1].axis("off")

        # pred contour
        axes[r, 2].imshow(img)
        draw_contours(axes[r, 2], pred, color="orange", linewidth=2.2)
        axes[r, 2].set_title(f"Pred Contour\nDice={s['dice']:.3f}", weight="bold")
        axes[r, 2].axis("off")

        # boundary error heatmap
        im1 = axes[r, 3].imshow(berr, cmap="coolwarm", vmin=-20, vmax=20)
        axes[r, 3].set_title("Boundary Error Map", weight="bold")
        axes[r, 3].axis("off")
        plt.colorbar(im1, ax=axes[r, 3], fraction=0.046, pad=0.04)

        # uncertainty
        im2 = axes[r, 4].imshow(unc, cmap="magma", vmin=0, vmax=1)
        axes[r, 4].set_title("Uncertainty Map", weight="bold")
        axes[r, 4].axis("off")
        plt.colorbar(im2, ax=axes[r, 4], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [ ]:
sorted_idx = np.argsort([s["dice"] for s in saved_samples])
worst_idx = sorted_idx[:3].tolist()
best_idx = sorted_idx[-3:].tolist()

plot_boundary_uncertainty_cases(saved_samples, worst_idx)
plot_boundary_uncertainty_cases(saved_samples, best_idx)

In [ ]:
def plot_class_heatstrip(results_df):
    class_order = ["nv", "mel", "bcc", "akiec", "bkl", "df", "vasc"]

    plt.figure(figsize=(12, 5))

    y_positions = np.arange(len(class_order))

    for i, cls in enumerate(class_order):
        vals = results_df.loc[results_df["class_name"] == cls, "dice"].values
        x = np.arange(len(vals))
        y = np.full_like(x, i, dtype=float)

        plt.scatter(
            vals,
            y,
            c=vals,
            cmap="viridis",
            vmin=0,
            vmax=1,
            s=28,
            alpha=0.85
        )

        plt.text(1.02, i, f"n={len(vals)}", va="center", fontsize=10)

    plt.yticks(y_positions, class_order)
    plt.xlabel("Dice Score")
    plt.title("Per-sample Dice Distribution by Class", fontsize=15, weight="bold")
    plt.xlim(0, 1.05)
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_class_heatstrip(results_df)

In [ ]:
def add_confidence_to_results(results_df, saved_samples):
    conf_map = {}
    for s in saved_samples:
        # average confidence over predicted lesion if available, else whole image
        pred = s["pred_mask"].astype(bool)
        prob = s["prob_map"]

        if pred.sum() > 0:
            mean_conf = float(prob[pred].mean())
        else:
            mean_conf = float(prob.mean())

        conf_map[s["image_id"]] = mean_conf

    df = results_df.copy()
    df["mean_pred_confidence"] = df["image_id"].map(conf_map)
    return df

results_with_conf = add_confidence_to_results(results_df, saved_samples)

plt.figure(figsize=(7, 5))
plt.scatter(
    results_with_conf["mean_pred_confidence"],
    results_with_conf["dice"],
    alpha=0.55
)
plt.xlabel("Mean Prediction Confidence")
plt.ylabel("Dice Score")
plt.title("Confidence vs Segmentation Quality", fontsize=15, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_area_agreement(results_df):
    df = results_df.copy()

    plt.figure(figsize=(7, 7))
    sc = plt.scatter(
        df["gt_area_pixels"],
        df["pred_area_pixels"],
        c=df["dice"],
        s=np.clip(df["abs_area_error"] / 20, 20, 250),
        alpha=0.6,
        cmap="viridis"
    )

    max_val = max(df["gt_area_pixels"].max(), df["pred_area_pixels"].max())
    plt.plot([0, max_val], [0, max_val], "--", linewidth=2, label="Perfect agreement")

    plt.xlabel("Ground-truth area (pixels)")
    plt.ylabel("Predicted area (pixels)")
    plt.title("Predicted Lesion Area vs Ground Truth", fontsize=14, weight="bold")
    plt.legend()
    plt.colorbar(sc, label="Dice score")
    plt.tight_layout()
    plt.show()

plot_area_agreement(results_df)

In [ ]:
def build_error_type_table(saved_samples):
    rows = []
    for s in saved_samples:
        gt = s["gt_mask"].astype(bool)
        pred = s["pred_mask"].astype(bool)

        tp = np.logical_and(gt, pred).sum()
        fp = np.logical_and(~gt, pred).sum()
        fn = np.logical_and(gt, ~pred).sum()

        rows.append({
            "image_id": s["image_id"],
            "class_name": inv_label_map[s["class_label"]],
            "tp": tp,
            "fp": fp,
            "fn": fn
        })
    return pd.DataFrame(rows)

error_df = build_error_type_table(saved_samples)

class_err = error_df.groupby("class_name")[["tp", "fp", "fn"]].mean()
class_err = class_err.div(class_err.sum(axis=1), axis=0)

ax = class_err.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)

plt.title("Average Error Composition by Class", fontsize=14, weight="bold")
plt.ylabel("Proportion")
plt.xlabel("Class")
plt.xticks(rotation=25)
plt.legend(["True Positive", "False Positive", "False Negative"])
plt.tight_layout()
plt.show()

In [ ]:
def plot_size_bin_performance(results_df):
    df = results_df.copy()

    df["size_bin"] = pd.qcut(
        df["gt_area_pixels"],
        q=4,
        labels=["Small", "Medium", "Large", "Very Large"]
    )

    grouped = df.groupby("size_bin")[["dice", "iou", "precision", "recall"]].mean()

    grouped.plot(kind="bar", figsize=(10, 5))
    plt.title("Performance Across Lesion Size Bins", fontsize=14, weight="bold")
    plt.ylabel("Score")
    plt.xlabel("Lesion Size Bin")
    plt.xticks(rotation=0)
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

plot_size_bin_performance(results_df)

In [ ]:
from matplotlib.colors import ListedColormap

def make_error_map(gt, pred):
    gt = gt.astype(bool)
    pred = pred.astype(bool)

    err = np.zeros_like(gt, dtype=np.uint8)
    err[(gt == 1) & (pred == 1)] = 1   # TP
    err[(gt == 0) & (pred == 1)] = 2   # FP
    err[(gt == 1) & (pred == 0)] = 3   # FN
    return err

def plot_random_error_sheet(saved_samples, n=12, seed=42):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(saved_samples), size=min(n, len(saved_samples)), replace=False)

    cmap = ListedColormap([
        (0, 0, 0, 1.0),         # TN black
        (0.0, 0.9, 0.5, 1.0),   # TP green
        (1.0, 0.5, 0.0, 1.0),   # FP orange
        (1.0, 0.15, 0.2, 1.0),  # FN red
    ])

    cols = 4
    rows = int(np.ceil(len(idxs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = np.array(axes).reshape(rows, cols)

    for ax in axes.flat:
        ax.axis("off")

    for ax, idx in zip(axes.flat, idxs):
        s = saved_samples[idx]
        err = make_error_map(s["gt_mask"], s["pred_mask"])
        ax.imshow(err, cmap=cmap, interpolation="nearest")
        ax.set_title(f"{s['class_label']} | Dice={s['dice']:.3f}", fontsize=10, weight="bold")
        ax.axis("off")

    plt.suptitle("Random Error Maps from the Test Set", fontsize=16, weight="bold")
    plt.tight_layout()
    plt.show()

plot_random_error_sheet(saved_samples, n=12)